# Run CESM-SMYLE benchmark preprocessing

This notebook is a lightweight driver for `run_process_cesm_smyle_benchmark.py`.

It passes the required settings to the preprocessing module and calls `process_one(...)` for selected fields, initialization months, years, ensemble members, and output frequencies.

Typical use cases:

- monthly benchmark for Niño3.4 SST skill: `fields = ["TS"]`, `init_months = [5, 11]`, `freqs = ["mon"]`
- seasonal benchmark for skill maps: `fields = ["TREFHT", "PRECT", "PSL"]`, `init_months = [5, 11]`, `freqs = ["seas"]`
- both monthly and seasonal benchmark files: `freqs = ["mon", "seas"]`


In [2]:
# Basic imports
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import xarray as xr

# Optional: Dask for parallel preprocessing
from dask.distributed import Client, LocalCluster
import dask

## 1. User settings

Edit this block for the dataset/variable you want to process.


In [3]:
# Path to the preprocessing script.
# If the .py file is in the same directory as this notebook, keep this as is.
script_path = Path("/global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py")

# Input/output paths
data_dir = "/global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE"
outdir = "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/"

# Variables and initialization months
# For Niño3.4 SST monthly skill, use TS.
fields = ["TS", "TREFHT", "PRECT", "PSL"]

# Usually May and November for the E3SM S2D comparison.
init_months = [2, 5, 8, 11]

# Choose benchmark frequencies to write.
#   "mon"  = monthly benchmark, needed for monthly Niño3.4 bottom panel
#   "seas" = seasonal benchmark, needed for seasonal skill maps/top panels
freqs = ["mon", "seas"]

# Years, members, and lead months
year_start = 1980
year_end = 2018
nens = 20
nlead = 24

# Processing behavior
force = False
dry_run = False
require_all_members = True
verify_coverage = True
run_verify = False

# Dask settings
use_dask = False
workers = 32
threads_per_worker = 1

print("script_path:", script_path)
print("fields:", fields)
print("init_months:", init_months)
print("freqs:", freqs)

script_path: /global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
fields: ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months: [2, 5, 8, 11]
freqs: ['mon', 'seas']


## 2. Import the preprocessing module

This imports the `.py` script as a module, so we can call `process_one(...)` directly from the notebook.


In [4]:
import importlib.util

if not script_path.exists():
    raise FileNotFoundError(
        f"Cannot find preprocessing script: {script_path}\n"
        "Put run_process_cesm_smyle_benchmark.py in the same directory "
        "as this notebook, or update script_path above."
    )

spec = importlib.util.spec_from_file_location("run_process_cesm_smyle_benchmark", script_path)
prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prep)

if outdir is None:
    outdir = prep.OUTDIR_DEFAULT

print("Imported module from:", script_path)
print("Benchmark outdir:", outdir)

Imported module from: /global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
Benchmark outdir: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/


## 3. Start Dask client

The preprocessing module uses Dask/xarray internally. Starting a local Dask client here helps parallelize loading, seasonal aggregation, and NetCDF writing.


In [5]:
client = None
cluster = None

if use_dask:
    dask.config.set({"array.slicing.split_large_chunks": True})
    cluster = LocalCluster(
        n_workers=workers,
        threads_per_worker=threads_per_worker,
    )
    client = Client(cluster)
    print("Dask dashboard:", client.dashboard_link)
else:
    print("Running without an explicit Dask distributed client.")

Running without an explicit Dask distributed client.


## 4. Build processing list

This mirrors the CLI logic from the `.py` script but keeps all settings visible in the notebook.


In [6]:
years = list(range(year_start, year_end + 1))
members = [f"EN{i:02d}" for i in range(1, nens + 1)]
combos = [(field, init_month) for field in fields for init_month in init_months]

print("=" * 70)
print("CESM-SMYLE benchmark preprocessing from notebook")
print("=" * 70)
print(f"data_dir      : {data_dir}")
print(f"outdir        : {outdir}")
print(f"fields        : {fields}")
print(f"init_months   : {init_months}")
print(f"freqs         : {freqs}")
print(f"years         : {years[0]}–{years[-1]}  ({len(years)} years)")
print(f"members       : EN01–EN{nens:02d}  ({nens} members)")
print(f"nlead         : {nlead} months")
print(f"combinations  : {len(combos)}")
print(f"force         : {force}")
print(f"dry_run       : {dry_run}")
print(f"run_verify    : {run_verify}")
print("=" * 70)

CESM-SMYLE benchmark preprocessing from notebook
data_dir      : /global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE
outdir        : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/
fields        : ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months   : [2, 5, 8, 11]
freqs         : ['mon', 'seas']
years         : 1980–2018  (39 years)
members       : EN01–EN20  (20 members)
nlead         : 24 months
combinations  : 16
force         : False
dry_run       : False
run_verify    : False


## 5. Run preprocessing

This calls:

```python
prep.process_one(...)
```

for each `(field, init_month)` combination.


In [7]:
%%time

total_t0 = time.perf_counter()
counters = {
    "ok": 0,
    "skipped": 0,
    "dry_run": 0,
    "no_data": 0,
    "failed": 0,
}

results = []

try:
    for field, init_month in combos:
        print("\n" + "-" * 70)
        print(f"Processing field={field}, init_month={init_month:02d}, freqs={freqs}")
        print("-" * 70)

        try:
            status = prep.process_one(
                field=field,
                init_month=init_month,
                data_dir=data_dir,
                outdir=outdir,
                years=years,
                members=members,
                nlead=nlead,
                require_all_members=require_all_members,
                verify_coverage=verify_coverage,
                force=force,
                dry_run=dry_run,
                run_verify=run_verify,
                freqs=freqs,
            )
            counters[status] = counters.get(status, 0) + 1

        except KeyboardInterrupt:
            print("\nInterrupted by user.")
            break

        except Exception as exc:
            warnings.warn(f"FAILED: field={field}, init_month={init_month}: {exc}")
            status = "failed"
            counters["failed"] += 1

        results.append(
            {
                "field": field,
                "init_month": init_month,
                "freqs": ",".join(freqs),
                "status": status,
            }
        )

finally:
    elapsed = time.perf_counter() - total_t0
    print("\n" + "=" * 70)
    print(f"Finished in {elapsed:.1f}s")
    print(f"Written:   {counters['ok']}")
    print(f"Skipped:   {counters['skipped']}")
    print(f"Dry-run:   {counters['dry_run']}")
    print(f"No data:   {counters['no_data']}")
    print(f"Failed:    {counters['failed']}")
    print("=" * 70)


----------------------------------------------------------------------
Processing field=TS, init_month=02, freqs=['mon', 'seas']
----------------------------------------------------------------------
  [SKIP]   BSMYLE02_TS_N20_M24_mon.nc, BSMYLE02_TS_N20_M24_seas.nc  (already exists)

----------------------------------------------------------------------
Processing field=TS, init_month=05, freqs=['mon', 'seas']
----------------------------------------------------------------------
  [SKIP]   BSMYLE05_TS_N20_M24_mon.nc, BSMYLE05_TS_N20_M24_seas.nc  (already exists)

----------------------------------------------------------------------
Processing field=TS, init_month=08, freqs=['mon', 'seas']
----------------------------------------------------------------------
  [SKIP]   BSMYLE08_TS_N20_M24_mon.nc, BSMYLE08_TS_N20_M24_seas.nc  (already exists)

----------------------------------------------------------------------
Processing field=TS, init_month=11, freqs=['mon', 'seas']
------------

## 6. Review outputs

This cell lists expected output files using the module's benchmark filename convention.


In [8]:
expected_files = []

for field in fields:
    for init_month in init_months:
        for freq in freqs:
            fname = prep.benchmark_filename(
                field,
                init_month,
                nens=nens,
                nlead=nlead,
                freq=freq,
            )
            expected_files.append(Path(outdir) / fname)

for path in expected_files:
    if path.exists():
        size_gb = path.stat().st_size / 1024**3
        print(f"[OK]      {path}  ({size_gb:.2f} GB)")
    else:
        print(f"[MISSING] {path}")

[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE02_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE02_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE05_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE05_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE08_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE08_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE11_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE11_TS_N20_M24_seas.nc  (0.61 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE02_TREFHT_N20_M24_mon.nc  (2.12 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE02_TREFHT_N20_M24_seas.nc  (0.62 GB)
[OK]      /global

## 7. Optional quick open check

Open one output file and inspect dimensions/metadata.


In [9]:
existing = [p for p in expected_files if p.exists()]

if existing:
    sample_file = existing[0]
    print("Opening:", sample_file)

    ds = xr.open_dataset(sample_file, chunks={})
    display(ds)
else:
    print("No output files found yet.")

Opening: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/BSMYLE02_TS_N20_M24_mon.nc


<xarray.Dataset> Size: 4GB
Dimensions:  (Y: 39, L: 24, M: 20, lat: 192, lon: 288)
Coordinates:
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * L        (L) int64 192B 1 2 3 4 5 6 7 8 9 10 ... 16 17 18 19 20 21 22 23 24
  * Y        (Y) <U10 2kB '1980020100' '1981020100' ... '2018020100'
  * M        (M) <U4 320B 'EN01' 'EN02' 'EN03' 'EN04' ... 'EN18' 'EN19' 'EN20'
Data variables:
    time     (Y, L) object 7kB dask.array<chunksize=(39, 24), meta=np.ndarray>
    TS       (Y, L, M, lat, lon) float32 4GB dask.array<chunksize=(1, 24, 1, 96, 144), meta=np.ndarray>
Attributes:
    source:        CESM-SMYLE hindcast data
    data_dir:      /global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE
    field:         TS
    init_month:    2
    nlead_months:  24
    n_members:     20
    year_range:    1980-2018
    grid:          f09_g17 (0.9x1.25 FV)
    frequency:     monthly
    processing:    monthly benchmark, no seasonal averaging
    created_by:    scripts/run_process_cesm_smyle_benchmark.py

## 8. Close Dask client

Run this after preprocessing is complete.


In [10]:
if client is not None:
    client.close()
    cluster.close()
    print("Closed Dask client and cluster.")